# ಪಾಠ 07 - ಯೋಜನೆ ವಿನ್ಯಾಸ ಮಾದರಿ

ಈ ನೋಟ್‌ಬುಕ್ ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್‌ವರ್ಕ್ ಅನ್ನು ಬಳಸಿ AI ಏಜೆಂಟ್‌ಗಳಿಗಾಗಿ **ಯೋಜನೆ ವಿನ್ಯಾಸ ಮಾದರಿ** ಅನ್ನು ಪ್ರದರ್ಶಿಸುತ್ತದೆ.
ನೀವು ಸಂಕೀರ್ಣ ಪ್ರಯಾಣ ವಿನಂತಿಗಳನ್ನು ರಚನಾತ್ಮಕ ಉಪಕಾರ್ಯಗಳಲ್ಲಿ ವಿಭಜಿಸುವುದು, ಅವುಗಳನ್ನು ತಜ್ಞ ಏಜೆಂಟ್‌ಗಳಿಗೆ ನಿಯೋಜಿಸುವುದು,
ಮತ್ತು ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಗಳಿಂದ ಶಕ್ತಿಪ್ರದರ್ಶಿತ ರಚನೆಗೊಳಿಸಿದ output ಬಳಸಿ ಫಲಿತಾಂಶ ಯೋಜನೆಯನ್ನು ಕಾರ್ಯಗತಗೊಳಿಸುವುದನ್ನು ಕಲಿಯುತ್ತೀರಿ.


## ಸ್ಥಾಪನೆ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ಕಾರ್ಯ ವಿಭಜನೆ

ಕಾರ್ಯ ವಿಭಜನೆ ಯೋಜನಾ ವಿನ್ಯಾಸ نمೂನೆಯ ಮುಖ್ಯಾಂಶವಾಗಿದೆ. ಒಂದೇ ಏಜೆಂಟ್‌ನ್ನು ಸಂಕೀರ್ಣ ವಿನಂತಿಯನ್ನು ಪೂರ್ಣವಾಗಿ ನಿರ್ವಹಿಸಲು ಕೇಳುವ ಬದಲು, ನಾವು ಸಮಸ್ಯೆಯನ್ನು ಸಣ್ಣ, ಚೆನ್ನಾಗಿ ವ್ಯಾಖ್ಯಾನಿಸಲಾದ **ಉಪಕಾರ್ಯಗಳಿಗೆ** ವಿಭಾಗಿಸುತ್ತೇವೆ. ಪ್ರತಿಯೊಂದು ಉಪಕಾರ್ಯವನ್ನು ಸ್ಪೆಷಲಿಸ್ಟ್ ಏಜೆಂಟ್‌ಗಳಿಗೆ (ಉದಾಹರಣೆಗೆ, ವಿಮಾನ, ಹೋಟೆಲ್ಗಳು, ಚಟುವಟಿಕೆಗಳು) ಸ್ಪಷ್ಟ ಪ್ರಾಧಾನ್ಯತೆ ಮತ್ತು ಅವಲಂಬನಾ ಕ್ರಮದೊಂದಿಗೆ ನೀಡಲಾಗುತ್ತದೆ.

ಈ ವಿಧಾನದ ಹಲವಾರು ಪ್ರಯೋಜನಗಳಿವೆ:
- **ಪೂರ್ಣತೆ**: ಪ್ರತಿಯೊಂದು ಉಪಕಾರ್ಯಕ್ಕೂ ಒಂದೇ ಹೊಣೆವಿಧಾನವಿದೆ.
- **ಸಮಾಂತರ**: ಸ್ವತಂತ್ರ ಉಪಕಾರ್ಯಗಳನ್ನು ಸಮಕಾಲೀನವಾಗಿ ನಡೆಸಬಹುದು.
- **ನಂಬಿಕೊಂಡಿರಲು ಸಾಧ್ಯತೆ**: ವೈಫಲ್ಯಗಳು ವೈಯಕ್ತಿಕ ಉಪಕಾರ್ಯಗಳಿಗೆ ಮಾತ್ರ ಪರಿಣಾಮ ಬೀರುತ್ತವೆ.
- **ಬಜೆಟ್ ಹಕ್ಕಿದಾರಿ**: ವೆಚ್ಚವನ್ನು ಪ್ರತಿ ಉಪಕಾರ್ಯಕ್ಕೆ ಅಂದಾಜಿಸಲಾಗುತ್ತದೆ ಮತ್ತು ಒಟ್ಟುಗೂಡಿಸಲಾಗುತ್ತದೆ.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## ಸಂರಚಿತ ಔಟ್ಪುಟ್ ಸಹಿತ ಯೋಜನಾ ಏಜೆಂಟ್ ರಚನೆ

ಯೋಜನಾ ಏಜೆಂಟ್ ಒಂದು **ಮುಖ್ಯಸ್ಥ ಚುನಾಯಿತರ್** ಆಗಿಯಾಗಿ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತದೆ. ಮೇಲಿನ ಮಟ್ಟದ ಪ್ರಯಾಣ ವಿನಂತಿ ನೀಡಿದಾಗ
ಅದು ಒಂದು ಸಂರಚಿತ `TravelPlan` ಅನ್ನು ಉತ್ಪಾದಿಸುತ್ತದೆ — ವಿನಂತಿಯನ್ನು ಉಪಕಾರ್ಯಗಳಿಗೆ ವಿಭಜಿಸಿ, ಆದ್ಯತೆಗಳನ್ನು ಸಿದ್ಧಮಾಡಿ,
ಮತ್ತು կախಿತತೆಗಳನ್ನು ಗುರುತಿಸಿ, ಆದಾಗ್ಯೂ ಒಂದು ಕಾನ್ಸಿಯರ್ಜ್ ಅಥವಾ ಕಾರ್ಯಗತಪಡಿಸಲು ಎನ್ನುವ ಹಂತವನ್ನು ನಿರ್ವಹಿಸಬಹುದು.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## ವಿಶೇಷ ಸಾಧನಗಳೊಂದಿಗೆ ಯೋಜನೆಯನ್ನು ನಿರ್ವಹಿಸುವುದು

ಒಂದು ಬಾರಿಗೆ ಫ್ರಂಟ್ ಡೆಸ್ಕ್ ಏಜೆಂಟ್ ಸಂಘಟಿತ ಯೋಜನೆಯನ್ನು ಸೃಷ್ಟಿಸಿದಾಗ, **ಕೋನ್ ಸಿಯರ್ಜ್ ಏಜೆಂಟ್** ಅದನ್ನು ನಿರ್ವಹಿಸುತ್ತದೆ. ಪ್ರತಿಯೊಂದು ವಿಶೇಷ ಸಾಧನವು ಸಹಾಯಕ ಕಾರ್ಯದ ಒಂದು ವರ್ಗವನ್ನು (ವಿಮಾನಗಳು, ಹೋಟೆಲ್‌ಗಳು, ಚಟುವಟಿಕೆಗಳು) ನಿಭಾಯಿಸುತ್ತದೆ. ಕೋನ್ ಸಿಯರ್ಜ್ ಯೋಜನೆಯ ಸಹಾಯಕ ಕಾರ್ಯಗಳನ್ನು ಅವಲಂಬನೆಯ ಕ್ರಮದಲ್ಲಿ ಪರಿಶೀಲಿಸಿ ಪ್ರತಿ ಕಾರ್ಯವನ್ನು ಸರಿಯಾದ ಸಾಧನಕ್ಕೆ ಕಳುಹಿಸುತ್ತದೆ.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು AI ಏಜೆಂಟ್‌ಗಳಿಗಾಗಿ **ಯೋಜನೆ ವಿನ್ಯಾಸ ಮಾದರಿ** ಅನ್ನು ಕಲಿತಿರಿ:

1. **ಕಾರ್ಯ ವಿಭಜನೆ** — ಫ್ರಂಟ್ ಡೆಸ್ಕ್ ಯೋಜನಾ ಏಜೆಂಟ್ ಸಂಕೀರ್ಣ ಪ್ರವಾಸ ವಿನಂತಿಯನ್ನು Pydantic ಮಾದರಿಗಳನ್ನು ಬಳಸಿಕೊಂಡು ರಚನೆಯಾದ ಉಪಕಾರ್ಯಗಳಾಗಿ ವಿಭಜಿಸಿ, ಪ್ರತಿ ಉಪಕಾರ್ಯವನ್ನು ವಿಶೇಷಜ್ಞ ಏಜೆಂಟ್‌ಗೆ ಪ್ರಾಥಮಿಕತೆ ಮತ್ತು ಅವಲಂಬನೆಗಳೊಂದಿಗೆ ನಿಗದಿಪಡಿಸುತ್ತದೆ.
2. **ರಚನೆಯಾದ خروج** — `response_format` ಅನ್ನು ಪాస్ ಮಾಡುವ ಮೂಲಕ ಏಜೆಂಟ್ ಮುಕ್ತರೂಪದ ಪಠ್ಯ ಬದಲಾಗಿ ದೃಢೀಕೃತ `TravelPlan` ವಸ್ತುವನ್ನು ಹಿಂದಿರುಗಿಸುತ್ತದೆ, ಇದರಿಂದ ನಂತರದ ಪ್ರಕ್ರಿಯೆ ವಿಶ್ವಾಸಾರ್ಹವಾಗುತ್ತದೆ.
3. **ಯೋಜನೆಯನ್ನು ಅನುಷ್ಠಾನಗೊಳಿಸುವುದು** — ಕಾನ್ಸಿಯರ್ಜ್ ಏಜೆಂಟ್ ವಿಶೇಷಜ್ಞ ಸಲಕರಣೆಗಳನ್ನು (`book_flight`, `reserve_hotel`, `book_activity`) ಬಳಸಿ ಉಪಕಾರ್ಯಗಳೊಳಗಿನ ಪ್ರಗತಿಯನ್ನು ಮತ್ತಲು ಯೋಜನೆಯನ್ನು ಅನುಷ್ಠಾನಗೊಳಿಸಿ ಫಲಿತಾಂಶಗಳನ್ನು ವರದಿ ಮಾಡುತ್ತದೆ.

ಈ ಮಾದರಿ *ಮಾಡಬೇಕಾಗಿರುವುದು* (ಯೋಜನೆ) ಮತ್ತು *ಅನ್ನುವಾಗುವ ವಿಧಾನ* (ಅನುಷ್ಠಾನ) ಅನ್ನು ವಿಭಜಿಸುತ್ತದೆ, ಏಜೆಂಟ್‌ಗಳನ್ನು ಹೆಚ್ಚಿನ ಮಾದರಿಯ, ಪರೀಕ್ಷಣೀಯ ಮತ್ತು ವಿಸ್ತಾರಗೊಳಿಸಲು ಸುಲಭವಾಗುವಂತೆ ಮಾಡುತ್ತದೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ತಪ್ಪು ಪರಿಹಾರ**:  
ಈ ಡಾಕ್ಯುಮೆಂಟ್ ಅನ್ನು AI ಅನುವಾದ ಸೇವೆಯಾದ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವದಿಸಲಾಗಿದೆ. ನಾವು ಶುದ್ಧತೆಗೆ ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ不ಮಿತಿ−ಗಳು ಇರಬಹುದು ಎಂದು ಗ akiyesiವಿರಲಿ. ಮೂಲ ಭಾಷೆಯಲ್ಲಿನ ಡಾಕ್ಯುಮೆಂಟ್ ಅನ್ನು ಅಧಿಕೃತ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದದ ಬಳಕೆಯಿಂದ ಸಂಭವಿಸಬಹುದಾದ ಯಾವುದೇ ಅವ್ಯಾಖ್ಯಾನಗಳು ಅಥವಾ ತಪ್ಪು ಅರ್ಥಮಾಡಿಕೊಳ್ಳಲು ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
